<a href="https://colab.research.google.com/github/BalakeerthiNidumolu/AyurBot/blob/main/ayurbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

DATASET_PATH = r"/content/drive/MyDrive/40"
IMG_SIZE = 224

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    subset="training"
)

val = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    subset="validation"
)

base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
out = Dense(train.num_classes, activation="softmax")(x)

model = Model(base.input, out)

model.compile(optimizer=Adam(0.0001), loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train, validation_data=val, epochs=10)

model.save("plant_mobilenetv2.h5")


Found 6945 images belonging to 30 classes.
Found 1722 images belonging to 30 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 2271s 10s/step - accuracy: 0.2203 - loss: 3.0313 - val_accuracy: 0.5168 - val_loss: 1.9181
Epoch 2/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 578s 2s/step - accuracy: 0.6770 - loss: 1.4219 - val_accuracy: 0.6446 - val_loss: 1.4051
Epoch 3/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 498s 2s/step - accuracy: 0.8273 - loss: 0.8923 - val_accuracy: 0.6864 - val_loss: 1.1997
Epoch 4/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 521s 2s/step - accuracy: 0.8742 - loss: 0.6493 - val_accuracy: 0.7050 - val_loss: 1.0928
Epoch 5/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 497s 2s/step - accuracy: 0.9104 - loss: 0.5061 - val_accuracy: 0.7224 - val_loss: 1.0185
Epoch 6/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 495s 2s/step - accuracy: 0.9276 - loss: 0.4039 - val_accuracy: 0.7340 - val_loss: 0.9697
Epoch 7/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 498s 2s/step - accuracy: 0.9356 - loss: 0.3434 - val_accuracy: 0.7439 - val_loss: 0.9465
Epoch 8/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 485s 2s/step - accuracy: 0.9471 - loss: 0.2990 - val_ac

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import json
import os

# ================= CONFIG =================
MODEL_PATH = "plant_mobilenetv2.h5"
LABEL_PATH = "/content/drive/MyDrive/class_labels1.json"
IMG_SIZE = 224

# ================= LOAD MODEL =================
model = tf.keras.models.load_model(MODEL_PATH)

# ================= LOAD LABELS =================
with open(LABEL_PATH, "r") as f:
    class_indices = json.load(f)

# Reverse mapping: index -> label
labels = {v: k for k, v in class_indices.items()}

# ================= PREDICTION FUNCTION =================
def predict_plant(image_path):
    # ---- Path check ----
    if not os.path.exists(image_path):
        return "Image not found", 0.0

    # ---- Read image ----
    img = cv2.imread(image_path)
    if img is None:
        return "Invalid image", 0.0

    # ---- Preprocess ----
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=0)

    # ---- Predict ----
    preds = model.predict(img, verbose=0)
    idx = int(np.argmax(preds))
    confidence = float(preds[0][idx] * 100)

    return labels[idx], confidence

# ================= TEST =================
if __name__ == "__main__":
    image_path = r"/content/drive/MyDrive/40/Coriender/136.jpg"
    plant, conf = predict_plant(image_path)

    print("🌱 Predicted Plant :", plant)
    print("✅ Confidence      :", round(conf, 2), "%")


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'plant_mobilenetv2.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)